# 24. Unsupervised Learning: K-Means Clustering

## Algorithm Category
**Type**: Unsupervised Learning - Clustering  
**Complexity**: Low-Medium  
**Use Case**: Partition data into k clusters based on similarity

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the K-Means algorithm and its mechanics
- Implement K-Means clustering from scratch and using scikit-learn
- Determine optimal number of clusters (elbow method, silhouette score)
- Visualize clusters and centroids
- Handle initialization and convergence issues
- Apply K-Means to real-world problems

## Historical Context

K-Means was developed by Stuart Lloyd in 1957:
- Lloyd, S.P. (1957): "Least squares quantization in PCM"
- One of the most popular clustering algorithms
- Simple, efficient, and widely applicable

**Key Papers/References:**
- Lloyd, S.P. (1957). "Least squares quantization in PCM"
- MacQueen, J. (1967). "Some methods for classification and analysis of multivariate observations"

## When to Use K-Means Clustering

K-Means is appropriate when:
- You know or can estimate the number of clusters
- Clusters are spherical and similar in size
- Data is numerical and continuous
- You need fast, scalable clustering
- Clusters are well-separated
- Working with large datasets

## Theory & Mechanics

### Mathematical Foundation

K-Means minimizes the within-cluster sum of squares (WCSS):

**Objective Function:**
$$J = \sum_{i=1}^{k} \sum_{x \in C_i} ||x - \mu_i||^2$$

Where:
- $k$: Number of clusters
- $C_i$: Set of points in cluster $i$
- $\mu_i$: Centroid of cluster $i$

**Algorithm Steps:**

1. **Initialize**: Randomly select $k$ centroids
2. **Assign**: Assign each point to nearest centroid
3. **Update**: Recalculate centroids as mean of assigned points
4. **Repeat**: Steps 2-3 until convergence (centroids don't change)

**Convergence:**
- Algorithm converges when centroids stabilize
- Guaranteed to converge (but may find local optimum)
- Typically converges in few iterations

### How It Works

1. **Initialization**: Choose k initial centroids (random or k-means++)
2. **Assignment**: For each point, find nearest centroid
3. **Update**: Move centroids to mean of assigned points
4. **Check**: If centroids changed, go to step 2; else stop

### Key Hyperparameters

- **n_clusters (k)**: Number of clusters to form
- **init**: Initialization method ('k-means++', 'random', or array)
- **n_init**: Number of times to run with different centroids
- **max_iter**: Maximum iterations per run
- **tol**: Tolerance for convergence
- **random_state**: Seed for reproducibility

### Advantages

- Simple and easy to understand
- Fast and efficient (O(nk) per iteration)
- Scales well to large datasets
- Works well with spherical clusters
- Guaranteed convergence

### Limitations

- Requires specifying number of clusters
- Sensitive to initialization (local optima)
- Assumes spherical clusters
- Sensitive to outliers
- Doesn't work well with non-convex clusters
- All clusters assumed to have similar size


## Implementation

Let's implement K-Means clustering.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

# Import our helper functions
from src.models.unsupervised import kmeans_cluster, evaluate_clustering
from src.processing.preprocessing import scale_features

print("Libraries imported successfully!")


In [ ]:
# Generate synthetic dataset
X, y_true = make_blobs(n_samples=300, centers=4, n_features=2, 
                       random_state=42, cluster_std=0.60)

print(f"Dataset Shape: {X.shape}")
print(f"True number of clusters: {len(np.unique(y_true))}")

# Visualize original data
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
plt.title('True Clusters')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)

# Apply K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X)

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=50, alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.title('K-Means Clustering (k=4)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nK-Means Results:")
print(f"  Number of clusters: {kmeans.n_clusters}")
print(f"  Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"  Number of iterations: {kmeans.n_iter_}")


## Finding Optimal Number of Clusters

Let's use the elbow method and silhouette score to find the optimal k.


In [ ]:
# Elbow Method
k_range = range(1, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)
    
    if k > 1:  # Silhouette score requires at least 2 clusters
        silhouette_avg = silhouette_score(X, kmeans.labels_)
        silhouette_scores.append(silhouette_avg)
    else:
        silhouette_scores.append(-1)

# Plot elbow curve and silhouette scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=4, color='r', linestyle='--', label='Optimal k=4')
axes[0].legend()

axes[1].plot(k_range, silhouette_scores, 'ro-')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=4, color='r', linestyle='--', label='Optimal k=4')
axes[1].legend()

plt.tight_layout()
plt.show()

optimal_k = k_range[np.argmax(silhouette_scores)]
print(f"Optimal number of clusters (by silhouette): {optimal_k}")
print(f"  Silhouette score: {max(silhouette_scores):.3f}")


## Validation & Testing

Let's validate the clustering and compare different k values.


In [ ]:
# Evaluate clustering
evaluation = evaluate_clustering(X, y_pred, algorithm='KMeans')
print("Clustering Evaluation:")
print(f"  Silhouette Score: {evaluation['silhouette_score']:.3f}")
print(f"  Number of clusters: {evaluation['n_clusters']}")
print(f"  Inertia: {kmeans.inertia_:.2f}")

# Compare different k values
k_values = [2, 3, 4, 5, 6]
comparison_results = []

for k in k_values:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans_test.fit_predict(X)
    sil_score = silhouette_score(X, labels)
    comparison_results.append({
        'k': k,
        'inertia': kmeans_test.inertia_,
        'silhouette': sil_score
    })
    print(f"\nk={k}:")
    print(f"  Inertia: {kmeans_test.inertia_:.2f}")
    print(f"  Silhouette: {sil_score:.3f}")

# Assertions
assert kmeans.n_clusters == 4, "Expected 4 clusters"
assert kmeans.inertia_ > 0, "Inertia should be positive"
assert evaluation['silhouette_score'] > 0, "Silhouette score should be positive"
print("\n✓ Validation checks passed")


## Silhouette Analysis

Let's perform detailed silhouette analysis.


In [ ]:
# Silhouette analysis for k=4
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_final = kmeans_final.fit_predict(X)

silhouette_vals = silhouette_samples(X, labels_final)

# Plot silhouette
fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(4):
    ith_cluster_silhouette_values = silhouette_vals[labels_final == i]
    ith_cluster_silhouette_values.sort()
    
    size_cluster_i = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i
    
    color = plt.cm.viridis(float(i) / 4)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values,
                     facecolor=color, edgecolor=color, alpha=0.7)
    
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
    y_lower = y_upper + 10

ax.set_xlabel('Silhouette Coefficient Values')
ax.set_ylabel('Cluster Label')
ax.set_title('Silhouette Plot for K-Means (k=4)')
ax.axvline(x=silhouette_score(X, labels_final), color="red", linestyle="--", 
           label=f'Average: {silhouette_score(X, labels_final):.3f}')
ax.legend()
plt.tight_layout()
plt.show()


## Real-World Application

Let's apply K-Means to the Iris dataset.


In [ ]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Scale features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply K-Means
kmeans_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
y_iris_pred = kmeans_iris.fit_predict(X_iris_scaled)

# Evaluate
sil_score_iris = silhouette_score(X_iris_scaled, y_iris_pred)
print("Iris Dataset Clustering:")
print(f"  Number of clusters: {kmeans_iris.n_clusters}")
print(f"  Silhouette Score: {sil_score_iris:.3f}")
print(f"  Inertia: {kmeans_iris.inertia_:.2f}")

# Visualize (using first two features)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', s=50, alpha=0.7)
plt.scatter(kmeans_iris.cluster_centers_[:, 0], kmeans_iris.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('True Labels')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris_pred, cmap='viridis', s=50, alpha=0.7)
plt.scatter(kmeans_iris.cluster_centers_[:, 0], kmeans_iris.cluster_centers_[:, 1],
           c='red', marker='x', s=200, linewidths=3, label='Centroids')
plt.xlabel(iris.feature_names[0])
plt.ylabel(iris.feature_names[1])
plt.title('K-Means Clustering (k=3)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **K-Means Basics**
   - Partition-based clustering algorithm
   - Minimizes within-cluster sum of squares
   - Requires specifying number of clusters
   - Iterative algorithm (assign, update, repeat)

2. **Algorithm Steps**
   - Initialize k centroids
   - Assign points to nearest centroid
   - Update centroids to mean of assigned points
   - Repeat until convergence

3. **Finding Optimal k**
   - **Elbow Method**: Plot inertia vs k, look for "elbow"
   - **Silhouette Score**: Measure cluster quality (-1 to 1, higher is better)
   - **Domain Knowledge**: Use prior knowledge about data

4. **Best Practices**
   - Scale features before clustering
   - Use k-means++ initialization (default)
   - Run multiple times with different seeds
   - Visualize results to validate clusters
   - Use silhouette analysis for validation

### When to Use K-Means

✅ **Good for:**
- Known or estimable number of clusters
- Spherical, well-separated clusters
- Numerical, continuous data
- Large datasets (scalable)
- When speed is important
- Exploratory data analysis

❌ **Not ideal for:**
- Unknown number of clusters
- Non-spherical clusters
- Clusters of different sizes
- Categorical data
- Outliers (sensitive)
- Non-convex cluster shapes

### Next Steps

- Try **K-Means++** initialization for better results
- Explore **Mini-Batch K-Means** for very large datasets
- Compare with **Hierarchical Clustering** for different cluster shapes
- Use **DBSCAN** for density-based clustering
- Apply **PCA** before clustering for high-dimensional data
